# 服务商品检索生成与匹配调试

这个 notebook 专门测试新版商品匹配逻辑。

当前检索只对 Excel 里的两个字段做 embedding：

- `服务商品名称`
- `关联故障现象`

测试顺序：

1. 检查配置和文件。
2. 单独测试 Qwen embedding API 是否可用。
3. 加载 Excel 服务商品。
4. 生成 `name/fault` 两组向量缓存。
5. 调用 `match_product_tool` 做匹配。

如果匹配结果为 `None`，先看 `status / error_code / message`，不要只看 `best_match`。

In [1]:
from pathlib import Path

from config.settings import settings

print("SPU_EXCEL_PATH:", settings.spu_excel_path)
print("excel_exists:", Path(settings.spu_excel_path).exists(), Path(settings.spu_excel_path).resolve())
print("EMBEDDING_CACHE_DIR:", settings.embedding_cache_dir)
print("QWEN_EMBEDDING_MODEL:", settings.qwen_embedding_model)
print("QWEN_EMBEDDING_BASE_URL:", settings.qwen_embedding_base_url)
print("has_qwen_key:", bool(settings.qwen_embedding_api_key))
print("has_openai_key_fallback:", bool(settings.openai_api_key))
print("match_threshold:", settings.product_match_threshold)
print("name_weight:", settings.product_name_weight)
print("fault_weight:", settings.product_fault_weight)

SPU_EXCEL_PATH: assets/spu.xlsx
excel_exists: True /Users/chengzongxin/project/hotel-ai-order/assets/spu.xlsx
EMBEDDING_CACHE_DIR: data/embedding_cache
QWEN_EMBEDDING_MODEL: text-embedding-v4
QWEN_EMBEDDING_BASE_URL: https://dashscope.aliyuncs.com/compatible-mode/v1
has_qwen_key: True
has_openai_key_fallback: True
match_threshold: 0.55
name_weight: 0.55
fault_weight: 0.45


## 1. 单独测试 Qwen embedding

这一步只发一个短文本到 embedding 接口。

如果这里失败，后面的缓存生成和匹配都会失败。常见原因：

- `QWEN_EMBEDDING_API_KEY` 不正确。
- `QWEN_EMBEDDING_MODEL` 不支持当前账号。
- `QWEN_EMBEDDING_BASE_URL` 配错。
- 网络代理拦截。

In [2]:
from rag.qwen_embedding import QwenEmbeddingClient

client = QwenEmbeddingClient()

try:
    test_embedding = client.embed_texts(["马桶堵塞"])
    print("embedding_shape:", test_embedding.shape)
    print("first_5_dims:", test_embedding[0][:5])
except Exception as exc:
    print("Qwen embedding 测试失败：")
    print(type(exc).__name__)
    print(exc)
    raise

embedding_shape: (1, 1024)
first_5_dims: [ 0.06720126  0.00767463  0.0704495  -0.01126234  0.03059202]


## 2. 加载 Excel 服务商品

这里不会调用 embedding API，只检查 Excel 数据是否加载正常。

In [3]:
from rag.spu_loader import SpuExcelLoader

records = SpuExcelLoader().load()
print("record_count:", len(records))

for record in records[:5]:
    print({
        "code": record.service_product_code,
        "name": record.service_product_name,
        "raw_service_type": record.raw_service_type,
        "service_order_type": record.service_order_type,
        "fault": record.fault_phenomenon,
    })

record_count: 454
{'code': 'FWSP01628', 'name': '双维修', 'raw_service_type': '维修', 'service_order_type': '托管维修', 'fault': '双故障'}
{'code': 'FWSP01627', 'name': '五金挂件（安装）', 'raw_service_type': '安装', 'service_order_type': '单次安装', 'fault': ''}
{'code': 'FWSP01623', 'name': 'LED屏', 'raw_service_type': '维修', 'service_order_type': '单次维修服务', 'fault': '屏幕不亮、无法上传字幕、无法更改内容或其他故障'}
{'code': 'FWSP01621', 'name': '消毒柜（大修）', 'raw_service_type': '维修', 'service_order_type': '单次维修服务', 'fault': '定时器不工作，臭氧发生器不工作'}
{'code': 'FWSP01620', 'name': '洗碗机（中修）', 'raw_service_type': '维修', 'service_order_type': '单次维修服务', 'fault': '无法进水，无法排水，漏水，水压不足，按键失灵'}


/Users/chengzongxin/project/hotel-ai-order/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## 3. 检查服务类型规则

服务类型不参与 embedding 文本，但会作为轻量规则加权。

In [4]:
from rag.product_matcher import ProductMatcher

retriever = ProductMatcher()

for text in ["马桶堵了", "帮我装五金挂件", "量一下窗帘尺寸", "托管维修"]:
    print(text, "=>", retriever.infer_service_type_hint(text))

马桶堵了 => 单次维修服务
帮我装五金挂件 => 单次安装
量一下窗帘尺寸 => 单次测量
托管维修 => 托管维修


## 4. 生成 name/fault 两组向量缓存

这一步会为所有服务商品生成两组 embedding：

- `name`：服务商品名称
- `fault`：关联故障现象

第一次运行会调用较多 embedding API。后续如果 Excel、模型和文本构建版本不变，会直接读取缓存。

In [5]:
name_embeddings = retriever.embeddings("name")
fault_embeddings = retriever.embeddings("fault")

print("name_embeddings:", name_embeddings.shape)
print("fault_embeddings:", fault_embeddings.shape)
print("record_count:", len(retriever.records))

name_embeddings: (454, 1024)
fault_embeddings: (454, 1024)
record_count: 454


## 5. 直接调用 Retriever

先绕过 Tool，直接看底层检索结果。这样更容易判断是检索问题还是 Tool 封装问题。

In [9]:
direct_cases = [
    {"query": "888房马桶堵了", "product": "马桶", "fault": "堵塞", "area": "卫生间"},
    {"query": "帮我装五金挂件", "product": "五金挂件", "fault": "安装", "area": "客房"},
    {"query": "量一下窗帘尺寸", "product": "窗帘", "fault": "测量", "area": "客房"},
]

for case in direct_cases:
    results = retriever.search(**case, top_k=3, threshold=0.0)
    print("\nQUERY:", case["query"])
    for item in results:
        print({
            "score": item["score"],
            "name_score": item["name_score"],
            "fault_score": item["fault_score"],
            "adjustment": item["service_type_adjustment"],
            "code": item["service_product_code"],
            "name": item["service_product_name"],
            "service_order_type": item["service_order_type"],
            "fault": item["fault_phenomenon"],
        })


QUERY: 888房马桶堵了
{'score': 0.618, 'name_score': 0.5434, 'fault_score': 0.5315, 'adjustment': 0.08, 'code': 'FWSP00883', 'name': '小便池疏通', 'service_order_type': '单次维修服务', 'fault': '堵塞，长流水，下水管漏水，下水慢，感应失灵'}
{'score': 0.5409, 'name_score': 0.4516, 'fault_score': 0.4722, 'adjustment': 0.08, 'code': 'FWSP01504', 'name': '蹲便器（大修）', 'service_order_type': '单次维修服务', 'fault': '蹲便器需更换'}
{'score': 0.5375, 'name_score': 0.365, 'fault_score': 0.5705, 'adjustment': 0.08, 'code': 'FWSP01618', 'name': '洗碗机（小修）', 'service_order_type': '单次维修服务', 'fault': '脏污，洗不干净，排水管堵塞，不通电'}

QUERY: 帮我装五金挂件
{'score': 0.6426, 'name_score': 0.8423, 'fault_score': 0.2208, 'adjustment': 0.08, 'code': 'FWSP01627', 'name': '五金挂件（安装）', 'service_order_type': '单次安装', 'fault': ''}
{'score': 0.6128, 'name_score': 0.5464, 'fault_score': 0.5161, 'adjustment': 0.08, 'code': 'FWSP01561', 'name': '钢化玻璃门五金(大修)', 'service_order_type': '单次安装', 'fault': '地弹簧损坏、五金全套坏、玻璃松动、安装位坏'}
{'score': 0.5971, 'name_score': 0.5812, 'fault_score': 0.4388, 'a

## 6. 调用 Tool

最后测试给大模型调用的 `match_product_tool`。

这里会打印完整返回结构：`status / error_code / message / data`。

In [7]:
from tools.product_match import match_product_tool

for case in direct_cases:
    result = match_product_tool.invoke({
        **case,
        "top_k": 3,
        "threshold": 0.45,
    })
    data = result.get("data", {})
    print("\nQUERY:", case["query"])
    print("status:", result.get("status"))
    print("error_code:", result.get("error_code"))
    print("message:", result.get("message"))
    print("service_type_hint:", data.get("service_type_hint"))
    print("count:", data.get("count"))
    print("best_match:")
    print(data.get("best_match"))

/Users/chengzongxin/project/hotel-ai-order/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



QUERY: 888房马桶堵了
status: success
error_code: None
message: ok
service_type_hint: 单次维修服务
count: 3
best_match:
{'service_product_code': 'FWSP00883', 'service_product_name': '小便池疏通', 'product_type': '普通商品', 'category': '卫浴水路/马桶', 'raw_service_type': '维修', 'service_order_type': '单次维修服务', 'unit': '起', 'price': '50.0', 'price_status': '正常', 'shelf_status': '上架', 'repair_category': '中修', 'related_category': 'PL7502-马桶', 'related_area': '', 'fault_phenomenon': '堵塞，长流水，下水管漏水，下水慢，感应失灵', 'display_order': '0.0', 'remark': '小便池疏通（疏通炮疏通、绞丝疏通）', 'score': 0.618, 'name_score': 0.5434, 'fault_score': 0.5315, 'service_type_adjustment': 0.08}

QUERY: 帮我装五金挂件
status: success
error_code: None
message: ok
service_type_hint: 单次安装
count: 3
best_match:
{'service_product_code': 'FWSP01627', 'service_product_name': '五金挂件（安装）', 'product_type': '普通商品', 'category': '墙面地面/五金挂件', 'raw_service_type': '安装', 'service_order_type': '单次安装', 'unit': '个', 'price': '25.0', 'price_status': '正常', 'shelf_status': '上架', 'repair_ca

## 7. 查看缓存文件

如果缓存生成成功，目录下应该出现：

- `product_name_*.npy`
- `product_name_*.json`
- `product_fault_*.npy`
- `product_fault_*.json`

In [10]:
cache_dir = Path(settings.embedding_cache_dir)
print("cache_dir:", cache_dir.resolve())

for path in sorted(cache_dir.glob("product_*")):
    print(path.name, path.stat().st_size, "bytes")

cache_dir: /Users/chengzongxin/project/hotel-ai-order/data/embedding_cache
service_product_fault_a61b7dc85b698fe3.json 289 bytes
service_product_fault_a61b7dc85b698fe3.npy 1859712 bytes
service_product_name_2868dcd7ca8ad83c.json 288 bytes
service_product_name_2868dcd7ca8ad83c.npy 1859712 bytes
